# Day 15: Two-Agent Handoff — Researcher Passes to Writer

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> One agent researches. Another writes. The handoff between them IS the product.

Multi-agent workflows start here. Today we build the simplest multi-agent pattern:
Agent A does research, then passes the result to Agent B who writes it up.

## What You'll Build
- A researcher agent that gathers facts
- A writer agent that turns facts into a blog post
- The handoff mechanism in all 3 frameworks

In [7]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 15 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 15 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Task

> **Write a short blog post about the benefits of AI agents for small businesses.**

Flow: Researcher gathers 3 key benefits → Writer turns them into a 150-word post.

---
## Framework 1: OpenAI Agents SDK — Handoff Chain

In [8]:
from agents import Agent, Runner, handoff, function_tool

model = get_openai_agents_model()

facts_db = [
    "AI agents automate repetitive tasks like customer support and data entry.",
    "Small businesses can use AI agents 24/7 without hiring additional staff.",
    "AI agents help small businesses compete with larger companies by scaling their capabilities.",
    "Cost-effective AI agents can handle scheduling, invoicing, and email management.",
    "AI agents improve response times, leading to higher customer satisfaction.",
]
search_round = 0

@function_tool
def search_benefits(query: str) -> str:
    """Search for facts about AI agent benefits for small businesses."""
    global search_round
    if search_round < len(facts_db):
        fact = facts_db[search_round]
        search_round += 1
        return f"Benefit #{search_round}: {fact}"
    return "No more facts."

researcher = Agent(
    name="Researcher",
    instructions=(
        "Research benefits of AI agents for small businesses. "
        "Use the search tool multiple times to find at least 3 benefits. "
        "When done, hand off to the Writer with your findings."
    ),
    tools=[search_benefits],
    model=model,
)

writer = Agent(
    name="Writer",
    instructions=(
        "Write a 150-word blog post based on the research findings. "
        "Make it engaging and accessible for small business owners. "
        "Include a catchy headline."
    ),
    model=model,
)

# Researcher hands off to Writer when done
researcher.handoffs.append(handoff(writer))

print("=" * 60)
print("OPENAI AGENTS SDK — RESEARCHER → WRITER HANDOFF")
print("=" * 60)

search_round = 0
result = await Runner.run(
    researcher,
    "Research AI agent benefits for small businesses and write a blog post.",
    max_turns=15,
)

# Print every step in the trace. result.final_output only shows the last
# agent's text, which hides whether the handoff actually fired. new_items
# exposes every RunItem with its producing agent and raw type, so a missing
# Writer output, an unfired handoff, or a content type mismatch all show up.
print(f"\n{'=' * 60}\nTRACE: {len(result.new_items)} items\n{'=' * 60}")
for i, item in enumerate(result.new_items):
    agent_name = getattr(item.agent, "name", "unknown")
    raw = item.raw_item
    raw_type = getattr(raw, "type", "unknown")
    print(f"\n[{i}] agent={agent_name} type={raw_type}")
    if hasattr(raw, "content") and raw.content:
        for c in raw.content:
            text = getattr(c, "text", None)
            if text:
                print(text)

print(f"\n{'=' * 60}\nFINAL OUTPUT\n{'=' * 60}")
print(result.final_output)

OPENAI AGENTS SDK — RESEARCHER → WRITER HANDOFF

TRACE: 6 items

[0] agent=Researcher type=function_call

[1] agent=Researcher type=unknown

[2] agent=Researcher type=message
Based on the initial search, here is one of the AI agent benefits identified for small businesses: they automate repetitive tasks such as customer support and data entry. Let's find more benefits to complete your blog post.



[3] agent=Researcher type=function_call

[4] agent=Researcher type=unknown

[5] agent=Researcher type=message
We found another benefit of using AI agents for small businesses: they operate non-stop, providing support without needing to hire extra staff. This allows your business to maintain a 24/7 service level even when you're not available.

Let's find more benefits now so we can complete your blog post.


FINAL OUTPUT
We found another benefit of using AI agents for small businesses: they operate non-stop, providing support without needing to hire extra staff. This allows your business to 

---
## Framework 2: LangGraph — Sequential Nodes

In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from typing_extensions import TypedDict, Annotated
import operator

model = get_langgraph_model()

facts_db = [
    "AI agents automate repetitive tasks like customer support.",
    "Small businesses can use AI agents 24/7 without extra hires.",
    "AI agents help small businesses compete with larger companies.",
    "Cost-effective AI agents handle scheduling, invoicing, and email.",
]
search_round = 0

@tool
def search_benefits(query: str) -> str:
    """Search for AI agent benefits."""
    global search_round
    if search_round < len(facts_db):
        f = facts_db[search_round]; search_round += 1
        return f"{f}"
    return "No more facts."

class PipelineState(TypedDict):
    messages: Annotated[list, operator.add]
    research_findings: str

def research_node(state: PipelineState) -> dict:
    r = model.invoke([
        SystemMessage(content="Research AI agent benefits for small businesses. Find 3 key benefits. List them clearly."),
        HumanMessage(content="Research AI agent benefits for small businesses."),
    ])
    return {"messages": [r], "research_findings": r.content}

def writing_node(state: PipelineState) -> dict:
    findings = state.get("research_findings", "")
    r = model.invoke([
        SystemMessage(content="Write a 150-word blog post. Include a headline. Make it engaging for small business owners."),
        HumanMessage(content=f"Based on this research, write a blog post:\n\n{findings}"),
    ])
    return {"messages": [r]}

builder = StateGraph(PipelineState)
builder.add_node("researcher", research_node)
builder.add_node("writer", writing_node)
builder.add_edge(START, "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", END)
graph = builder.compile()

print("=" * 60)
print("LANGGRAPH — RESEARCHER → WRITER (sequential nodes)")
print("=" * 60)

search_round = 0
result = graph.invoke({"messages": [HumanMessage(content="Go")], "research_findings": ""})
print("\nRESEARCH:")
print(result["research_findings"][:300])
print("\nBLOG POST:")
print(result["messages"][-1].content)

LANGGRAPH — RESEARCHER → WRITER (sequential nodes)

RESEARCH:
When researching the benefits of using AI agents for small businesses, several key advantages become apparent:

1. **Increased Efficiency and Productivity**: AI agents can handle routine tasks such as customer service inquiries, scheduling appointments, or data entry more quickly than human employee

BLOG POST:
### Revolutionizing Your Business: How AI Agents Can Supercharge Small Businesses

Are you ready to take your small business to the next level? One powerful tool that's quietly revolutionizing operations is Artificial Intelligence (AI) agents. Let’s explore how implementing these digital helpers can significantly boost efficiency, enhance customer service, and save costs without breaking the bank.

#### Increased Efficiency and Productivity
One of the most compelling benefits of AI agents is their ability to handle mundane tasks like customer inquiries, scheduling appointments, or data entry with lightning speed. By a

---
## Framework 3: CrewAI — Sequential Tasks

In [4]:
from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

researcher = Agent(
    role="Research Analyst",
    goal="Find 3 key benefits of AI agents for small businesses",
    backstory="You are a thorough researcher who finds actionable insights.",
    llm=llm, verbose=True,
)

writer = Agent(
    role="Blog Writer",
    goal="Turn research into an engaging 150-word blog post",
    backstory="You write catchy headlines and engaging content for small business owners.",
    llm=llm, verbose=True,
)

research_task = Task(
    description="Research 3 key benefits of AI agents for small businesses. List them clearly.",
    expected_output="A list of 3 benefits with brief explanations.",
    agent=researcher,
)

writing_task = Task(
    description="Write a 150-word blog post with a catchy headline based on the research findings.",
    expected_output="A 150-word blog post with headline.",
    agent=writer,
    context=[research_task],  # THIS is the handoff — writer sees research output
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    verbose=True,
)

print("=" * 60)
print("CREWAI — RESEARCHER → WRITER (sequential tasks)")
print("=" * 60)

result = await crew.kickoff_async()
print(f"\n{result}")

CREWAI — RESEARCHER → WRITER (sequential tasks)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.6                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 16045352-e59e-4d37-a901-2e1ad96629c9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research 3 key benefits of AI agents for small businesses. List them clearly.                            │
│  ID: 8fe754d9-bb20-412a-ba30-eab6e2ef4a23                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Task: Research 3 key benefits of AI agents for small businesses. List them clearly.                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Blog Writer                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Boost Your Small Business Skyrocketing Efficiency & Satisfaction With AI Agents                            │
│                                                                                                                 │
│  AI chatbots are turning tables for small businesses by revolutionizing customer service, marketing, and        │
│  employee productivity. Equipped with natural language processing capabilities, AI-driven chatbots ensure       │
│  instant, consistent support around the clock. They handle common inquiries swiftly like a well-oiled machine,  │
│  significantly boosting customer satisfaction and reducing wait times.                                          │
│                                                                                                                 │
│  But these agents do more—they’re game-changing for marketing too! With their ability to analyze colossal data  │
│  volumes at lightning speed, AI tools help small businesses optimize advertising campaigns and generate         │
│  personalized recommendations based on real-time consumer behavior. This yields higher conversion rates and     │
│  streamlined sales processes.                                                                                   │
│                                                                                                                 │
│  And it’s not just about customer service. By automating repetitive tasks, AI agents drastically reduce         │
│  operational costs—leaving employees fresher, more engaged with their core responsibilities. They facilitate    │
│  better productivity and morale within teams, allowing managers to focus on developing new ideas that will      │
│  drive long-term growth rather than mundane management duties.                                                  │
│                                                                                                                 │
│  Upgrade your business now by integrating AI into your operations for a smarter, more efficient tomorrow!       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a 150-word blog post with a catchy headline based on the research findings.                        │
│  Agent: Blog Writer                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 16045352-e59e-4d37-a901-2e1ad96629c9                                                                       │
│  Final Output: ### Boost Your Small Business Skyrocketing Efficiency & Satisfaction With AI Agents              │
│                                                                                                                 │
│  AI chatbots are turning tables for small businesses by revolutionizing customer service, marketing, and        │
│  employee productivity. Equipped with natural language processing capabilities, AI-driven chatbots ensure       │
│  instant, consistent support around the clock. They handle common inquiries swiftly like a well-oiled machine,  │
│  significantly boosting customer satisfaction and reducing wait times.                                          │
│                                                                                                                 │
│  But these agents do more—they’re game-changing for marketing too! With their ability to analyze colossal data  │
│  volumes at lightning speed, AI tools help small businesses optimize advertising campaigns and generate         │
│  personalized recommendations based on real-time consumer behavior. This yields higher conversion rates and     │
│  streamlined sales processes.                                                                                   │
│                                                                                                                 │
│  And it’s not just about customer service. By automating repetitive tasks, AI agents drastically reduce         │
│  operational costs—leaving employees fresher, more engaged with their core responsibilities. They facilitate    │
│  better productivity and morale within teams, allowing managers to focus on developing new ideas that will      │
│  drive long-term growth rather than mundane management duties.                                                  │
│                                                                                                                 │
│  Upgrade your business now by integrating AI into your operations for a smarter, more efficient tomorrow!       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


### Boost Your Small Business Skyrocketing Efficiency & Satisfaction With AI Agents

AI chatbots are turning tables for small businesses by revolutionizing customer service, marketing, and employee productivity. Equipped with natural language processing capabilities, AI-driven chatbots ensure instant, consistent support around the clock. They handle common inquiries swiftly like a well-oiled machine, significantly boosting customer satisfaction and reducing wait times.

But these agents do more—they’re game-changing for marketing too! With their ability to analyze colossal data volumes at lightning speed, AI tools help small businesses optimize advertising campaigns and generate personalized recommendations based on real-time consumer behavior. This yields higher conversion rates and streamlined sales processes. 

And it’s not just about customer service. By automating repetitive tasks, AI agents drastically reduce operational costs—leaving employees fresher, more engaged with their c

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Comparison: Handoff Mechanisms

| Framework | Handoff mechanism | Data passing | Visibility |
|---|---|---|---|
| OpenAI SDK | `handoff()` function | Automatic (via conversation) | In Runner trace |
| LangGraph | Edge from node A to node B | Via shared state | Full graph trace |
| CrewAI | `context=[task]` on next task | Via task output | In verbose output |

**Key insight:** CrewAI's `context=[task]` is the cleanest handoff API — it explicitly
connects one task's output to the next task's input. LangGraph uses shared state.
OpenAI SDK uses the conversation as the data channel.

## Key Takeaway

The researcher-writer handoff is the fundamental multi-agent pattern. All three
frameworks handle it well, but the mechanism differs:
- CrewAI: `context=[task]` — most explicit data passing
- LangGraph: shared state — most flexible
- OpenAI SDK: handoff() — most automatic

---

